## Table 2: DeiT-Base model's class leakage results on CIFAR10 dataset using RTX 4500 Ada GPU

In [1]:
import numpy as np
import pandas as pd

### Section 1: Loading Dataset

In [ ]:
all_class_data=[]
for i in range(1,501):
    temp=[]
    for j in range(10):
        path='./mem_csv_deit_base_cifar10_class/report'+str(j)+'_'+str(i)+'.csv'
        data=pd.read_csv(path)
        temp.append(data)
    all_class_data.append(temp)
    
print(all_class_data[0][1].columns)

Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'smsp__warps_active.min.peak_sustained',
       'smsp__warps_active.min.per_cycle_active',
       'smsp__warps_active.sum.peak_sustained',
       'smsp__warps_active.sum.per_cycle_active',
       'smsp__warps_eligible.avg.per_cycle_active',
       'smsp__warps_eligible.max.per_cycle_active',
       'smsp__warps_eligible.min.per_cycle_active',
       'smsp__warps_eligible.sum.per_cycle_active', 'thread_inst_executed',
       'thread_inst_executed_true'],
      dtype='object', length=1220)


### Section 2: Pre-processing Dataset

In [3]:
for i in range(500):
    for j in range(10):
        all_class_data[i][j]=all_class_data[i][j].drop(['ID'], axis=1)
        if 0 in all_class_data[i][j].index:
            all_class_data[i][j] = all_class_data[i][j].drop(index=0)

In [4]:
print(all_class_data[0][1].columns)

Index(['Process ID', 'Process Name', 'Host Name', 'Kernel Name', 'Context',
       'Stream', 'Block Size', 'Grid Size', 'Device', 'CC',
       ...
       'smsp__warps_active.min.peak_sustained',
       'smsp__warps_active.min.per_cycle_active',
       'smsp__warps_active.sum.peak_sustained',
       'smsp__warps_active.sum.per_cycle_active',
       'smsp__warps_eligible.avg.per_cycle_active',
       'smsp__warps_eligible.max.per_cycle_active',
       'smsp__warps_eligible.min.per_cycle_active',
       'smsp__warps_eligible.sum.per_cycle_active', 'thread_inst_executed',
       'thread_inst_executed_true'],
      dtype='object', length=1219)


### Section 3: Metric-based Data filtering and pre-processing

In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib


kernel_name=['fmha_cutlassF_f32_aligned_64x64_rf_sm80']
ltx_cols = [col for col in all_class_data[0][1].columns if col.startswith('thread_inst_executed')]# or col.startswith('gpu__') or col.startswith('sm__')]
for col in ltx_cols:
    for i in range(500):
        for j in range(10):
            if all_class_data[i][j][col].dtype == 'object':  # likely string
                all_class_data[i][j][col] = pd.to_numeric(all_class_data[i][j][col].astype(str).str.replace(',', '').str.strip(), errors='coerce')
        

print(len(all_class_data))

### Section 4: Generating classifier model's training data and validation data by considering only the filtered Metrics.

In [6]:
col = [col for col in all_class_data[0][1].columns if 
        col.startswith('smsp__branch_targets_threads_divergent') 
        or col.startswith('smsp__inst_executed.sum') 
        or col.startswith('smsp__inst_executed_op_shared_atom') 
         or col.startswith('smsp__sass_thread_inst_executed_op_fadd_pred_on.min') #
        or col.startswith('smsp__sass_thread_inst_executed_op_ffma_pred_on.min') #
        or col.startswith('smsp__sass_thread_inst_executed_op_fmul_pred_on.min') #
    or col.startswith('dram__bytes_write.sum')
]

col.append('thread_inst_executed')

for i in range(500):
    for j in range(10):
        for c in col:
            all_class_data[i][j][c]=pd.to_numeric(all_class_data[i][j][c].astype(str).str.replace(',', '').str.strip(), errors='coerce').astype('float32')

# Loop through each DataFrame and keep only the selected columns (if they exist)
df=[[[] for i in range(10)] for j in range(500)]
for i in range(len(all_class_data)):
    for j in range(len(all_class_data[i])):
        df2 = all_class_data[i][j]
        # Keep only the columns that exist in the current DataFrame
        selected_cols = [c for c in col if c in df2.columns]
        df[i][j] = df2[selected_cols].copy()

### Dataset re-arrangement

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X=[]
y=[]
for i in range(500):
    for j in range(10):
        t=scaler.fit_transform(df[i][j])
        # t=df[i][j]
        X.append(t)
        y.append(j)
X = np.array(X)
y = np.array(y)
print(X.shape, y.shape)

(5000, 12, 11) (5000,)


In [8]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
inp_ch= X.shape[2]

cuda:0


### Attack Classifier training and 5-fold cross validation

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNTimeSeriesClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNTimeSeriesClassifier, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=inp_ch, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # Input shape: (batch_size, 12, 6)
        x = x.permute(0, 2, 1)  # Convert to (batch_size, 6, 12)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)
        x = self.fc(x)  # (batch_size, num_classes)
        return x


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
from torch.optim.lr_scheduler import ReduceLROnPlateau



X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, y_train.shape)
# Move to DataLoader if needed
batch_size = 32
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
val_dataset = torch.utils.data.TensorDataset(X_val, y_val)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size)

torch.Size([4000, 12, 11]) torch.Size([4000])


In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import random

random.seed(10)

# Configuration
k_folds = 5
num_epochs = 85
batch_size = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Assuming X and y are torch tensors
# X = X.cpu().numpy()  # Convert to numpy for sklearn
# y = y.cpu().numpy()

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")

    # Create data for current fold
    X_train_fold = X[train_idx].clone().detach().float()
    y_train_fold = y[train_idx].clone().detach().long()
    X_val_fold = X[val_idx].clone().detach().float()
    y_val_fold = y[val_idx].clone().detach().long()

    train_dataset = TensorDataset(X_train_fold, y_train_fold)
    val_dataset = TensorDataset(X_val_fold, y_val_fold)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # New model instance per fold
    model = CNNTimeSeriesClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)#, verbose=True)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        y_true_train, y_pred_train = [], []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            y_pred_train.extend(torch.argmax(out, dim=1).cpu().numpy())
            y_true_train.extend(yb.cpu().numpy())

        train_acc = accuracy_score(y_true_train, y_pred_train)

        # Validation
        model.eval()
        y_true_val, y_pred_val = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                y_pred_val.extend(torch.argmax(out, dim=1).cpu().numpy())
                y_true_val.extend(yb.cpu().numpy())

        val_acc = accuracy_score(y_true_val, y_pred_val)
        
        print(f"Fold {fold} | Epoch {epoch:02d} | Loss: {total_loss:.4f} "
              f"| Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    # Store final accuracy for this fold
    fold_results.append(val_acc)

# Summary
print("\n=== K-Fold Cross Validation Results ===")
print(f"Accuracies for each fold: {[f'{acc*100:.2f}%' for acc in fold_results]}")
print(f"Average Accuracy: {np.mean(fold_results)*100:.2f}%")



--- Fold 1 ---
Fold 1 | Epoch 01 | Loss: 256.1228 | Train Acc: 30.58% | Val Acc: 42.10%
Fold 1 | Epoch 02 | Loss: 196.3588 | Train Acc: 50.45% | Val Acc: 54.40%
Fold 1 | Epoch 03 | Loss: 165.7153 | Train Acc: 58.77% | Val Acc: 57.40%
Fold 1 | Epoch 04 | Loss: 145.6909 | Train Acc: 63.20% | Val Acc: 62.40%
Fold 1 | Epoch 05 | Loss: 130.2886 | Train Acc: 67.15% | Val Acc: 66.50%
Fold 1 | Epoch 06 | Loss: 117.6527 | Train Acc: 70.67% | Val Acc: 70.10%
Fold 1 | Epoch 07 | Loss: 105.7566 | Train Acc: 73.35% | Val Acc: 71.10%
Fold 1 | Epoch 08 | Loss: 98.4858 | Train Acc: 75.60% | Val Acc: 73.20%
Fold 1 | Epoch 09 | Loss: 91.2559 | Train Acc: 77.12% | Val Acc: 74.70%
Fold 1 | Epoch 10 | Loss: 86.2725 | Train Acc: 78.72% | Val Acc: 75.40%
Fold 1 | Epoch 11 | Loss: 81.5793 | Train Acc: 80.35% | Val Acc: 75.90%
Fold 1 | Epoch 12 | Loss: 76.3311 | Train Acc: 81.77% | Val Acc: 78.20%
Fold 1 | Epoch 13 | Loss: 72.6983 | Train Acc: 82.33% | Val Acc: 77.80%
Fold 1 | Epoch 14 | Loss: 68.8634 | Train